# LmrR — Complete QM Benchmark Pipeline (FIXED)

Self-contained notebook. It does not use `__file__` and does not depend on separate `.py` files.

In [1]:

from pathlib import Path
import csv
import re

# ============================================================
# LmrR QM BENCHMARK PIPELINE — SELF-CONTAINED JUPYTER VERSION
# ============================================================
# This notebook does NOT use __file__.
# It prepares ORCA input files; it does not run ORCA.
#
# Reference:
#   B3LYP-D3BJ/def2-SVP optimization + frequencies
#
# Benchmarks:
#   PBE0-D3BJ/def2-SVP SP
#   PBE0-D3BJ/def2-TZVP SP
#   omegaB97M-V/def2-TZVP SP
#   omegaB97X-V/def2-TZVP SP
#   PBE0-DH/def2-TZVP SP
#
# Output root:
#   D:\PhD_Thesis\LmrR_EVB\charges\qm_reference_for_evb
# ============================================================

BASE_DIR = Path(r"D:\PhD_Thesis\LmrR_EVB")
CHARGES_DIR = BASE_DIR / "charges"
OUT_DIR = CHARGES_DIR / "qm_reference_for_evb"

NPROCS = 8
MAXCORE_MB = 3000
CHARGE = 0
MULTIPLICITY = 1

# Exact reaction/state mapping used by the finalized pipeline.
STRUCTURE_SETS = {
    "RS1_1_to_TS1_2": ("step_1_1_RS.pdb", "step_1_1_TS.pdb", "step_1_1_PS.pdb"),
    "TS1_2_to_PS1_2b": ("step_1_2_RS.pdb", "step_1_2_TS.pdb", "step_1_2_PS.pdb"),
    "RS1_2b_to_PS1_3": ("step_1_3_RS.pdb", "step_1_3_TS.pdb", "step_1_3_PS.pdb"),
    "RS2_1_to_TS2_1a": ("step_2_1_RS.pdb", "step_2_1_TS.pdb", "step_2_1_PS.pdb"),
    "TS2_1a_to_PS2_2": ("step_2_2a_RS.pdb", "step_2_2b_TS.pdb", "step_2_2b_PS.pdb"),
}

STATES = ("RS", "TS", "PS")

def mkdir(path):
    path.mkdir(parents=True, exist_ok=True)

def parse_pdb(path):
    atoms = []
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.startswith(("ATOM  ", "HETATM")):
            continue
        element = line[76:78].strip()
        if not element:
            letters = "".join(c for c in line[12:16] if c.isalpha())
            element = (
                "Cl" if letters[:2].lower() == "cl" else
                "Br" if letters[:2].lower() == "br" else
                (letters[:1] or "X")
            )
        atoms.append({
            "serial": int(line[6:11]),
            "name": line[12:16].strip(),
            "resname": line[17:20].strip(),
            "chain": line[21:22].strip(),
            "resid": line[22:26].strip(),
            "element": element.capitalize(),
            "x": float(line[30:38]),
            "y": float(line[38:46]),
            "z": float(line[46:54]),
        })
    if not atoms:
        raise ValueError(f"No ATOM/HETATM records found in {path}")
    return atoms

def write_xyz_and_mapping(pdb, xyz, mapping):
    atoms = parse_pdb(pdb)
    mkdir(xyz.parent)
    with xyz.open("w", encoding="utf-8") as f:
        f.write(f"{len(atoms)}\n{pdb.name}\n")
        for a in atoms:
            f.write(
                f"{a['element']:2s} "
                f"{a['x']:14.8f} "
                f"{a['y']:14.8f} "
                f"{a['z']:14.8f}\n"
            )
    with mapping.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["xyz_index","pdb_serial","resid","resname","atom_name","element"])
        for i, a in enumerate(atoms, 1):
            w.writerow([i,a["serial"],a["resid"],a["resname"],a["name"],a["element"]])

def write_orca_input(path, route, geometry):
    mkdir(path.parent)
    text = f"""! {route}

%pal
   nprocs {NPROCS}
end

%maxcore {MAXCORE_MB}

%output
   Print[P_Mulliken] 1
end

* xyzfile {CHARGE} {MULTIPLICITY} {geometry}
"""
    path.write_text(text, encoding="utf-8")

def state_dir(method, label, state):
    p = OUT_DIR / method / label / state
    for sub in ("input", "geometry", "energy", "charges", "frequencies"):
        mkdir(p / sub)
    return p

# ------------------------------------------------------------
# 1. B3LYP-D3BJ/def2-SVP reference
# ------------------------------------------------------------

for label, pdbs in STRUCTURE_SETS.items():
    for state, pdb_name in zip(STATES, pdbs):
        pdb = CHARGES_DIR / pdb_name
        if not pdb.exists():
            print(f"WARNING: missing PDB: {pdb}")
            continue

        sd = state_dir("B3LYP_def2SVP", label, state)
        xyz = sd / "geometry" / f"{label}_{state}_initial.xyz"
        mapping = sd / "geometry" / f"{label}_{state}_pdb_to_xyz_map.csv"
        write_xyz_and_mapping(pdb, xyz, mapping)

        route = (
            "B3LYP D3BJ def2-SVP OptTS NumFreq TightSCF"
            if state == "TS"
            else
            "B3LYP D3BJ def2-SVP Opt NumFreq TightSCF"
        )

        write_orca_input(
            sd / "input" / f"{label}_{state}_B3LYP_def2SVP.inp",
            route,
            xyz,
        )

print("B3LYP/def2-SVP reference inputs prepared.")

# ------------------------------------------------------------
# Find optimized B3LYP geometry for benchmark SP calculations
# ------------------------------------------------------------

def find_optimized_geometry(label, state):
    candidates = [
        OUT_DIR / "B3LYP_def2SVP" / label / state / "geometry" / f"{label}_{state}_B3LYP_def2SVP_opt.xyz",
        OUT_DIR / "B3LYP_def2SVP" / label / state / "geometry" / f"{label}_{state}_DFT_opt.xyz",
        OUT_DIR / "B3LYP_def2SVP" / label / state / "geometry" / f"{label}_{state}_opt.xyz",
        CHARGES_DIR / "fast_lmrr_evb_qm" / "pdb_preserving_xtb_qm_inputs" / label / f"{label}_{state}_xtbopt.xyz",
        CHARGES_DIR / "fast_lmrr_evb_qm" / "pdb_preserving_xtb_qm_inputs" / label / f"{label}_{state}.xyz",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

# ------------------------------------------------------------
# 2–6. Benchmark single points
# ------------------------------------------------------------

BENCHMARKS = {
    "PBE0_def2SVP": (
        "PBE0 D3BJ def2-SVP SP TightSCF",
        "PBE0 hybrid functional / def2-SVP"
    ),
    "PBE0_def2TZVP": (
        "PBE0 D3BJ def2-TZVP SP TightSCF",
        "PBE0 hybrid functional / def2-TZVP"
    ),
    "wB97M-V_def2TZVP": (
        "wB97M-V def2-TZVP SP TightSCF",
        "omegaB97M-V range-separated hybrid meta-GGA with VV10 nonlocal correlation / def2-TZVP"
    ),
    "wB97X-V_def2TZVP": (
        "wB97X-V def2-TZVP SP TightSCF",
        "omegaB97X-V range-separated hybrid with VV10 nonlocal correlation / def2-TZVP"
    ),
    "PBE0-DH_def2TZVP": (
        "PBE0-DH def2-TZVP SP TightSCF",
        "PBE0-DH double-hybrid functional / def2-TZVP"
    ),
}

for method, (route, description) in BENCHMARKS.items():
    for label in STRUCTURE_SETS:
        for state in STATES:
            geometry = find_optimized_geometry(label, state)
            if geometry is None:
                print(f"WARNING: optimized geometry not found for {label}/{state}")
                continue

            sd = state_dir(method, label, state)
            write_orca_input(
                sd / "input" / f"{label}_{state}_{method}_SP.inp",
                route,
                geometry,
            )
            (sd / "README.txt").write_text(
                description + "\nSingle-point benchmark on the reference geometry.\n",
                encoding="utf-8"
            )

    print(f"Prepared: {method}")

print("\nALL QM INPUTS PREPARED.")
print(f"Output directory: {OUT_DIR}")
print("No ORCA calculations were launched.")
print("Run B3LYP/def2-SVP first, then the benchmark single points.")
print("Test PBE0-DH/def2-TZVP on one representative structure before the full set.")


B3LYP/def2-SVP reference inputs prepared.
Prepared: PBE0_def2SVP
Prepared: PBE0_def2TZVP
Prepared: wB97M-V_def2TZVP
Prepared: wB97X-V_def2TZVP
Prepared: PBE0-DH_def2TZVP

ALL QM INPUTS PREPARED.
Output directory: D:\PhD_Thesis\LmrR_EVB\charges\qm_reference_for_evb
No ORCA calculations were launched.
Run B3LYP/def2-SVP first, then the benchmark single points.
Test PBE0-DH/def2-TZVP on one representative structure before the full set.
